# HW3 · 04 — Ensembles and macro-F1 tuning (own experiments)
Inputs: probability files saved by notebooks 02 (TF-IDF LogReg) and 03 (XLM-R).
Kaggle: attach both notebooks via Add Input. Colab: put `baseline_val_proba.csv` + `baseline_test_proba.csv`
(from the Output tab of notebook 02) into Google Drive `hw3_outputs/`, next to the files written by notebook 03.
Fixed reference = XLM-R alone (best inference mode from notebook 03).

Everything that is *tuned* here (blend weight, class offsets, a stacking model) is tuned on the val split,
so to get an honest score we use **5-fold cross-validation inside val**: tune on 4 folds, score the 5th,
and compute macro-F1 on the concatenated out-of-fold predictions. Train/test rows are never used for tuning.

Experiments:
* **E1 blend**  — weighted average of XLM-R and TF-IDF probabilities
* **E2 offsets** — per-class log-probability offsets (a threshold for macro-F1; helps the hard *fusion* class)
* **E3 stacking** — logistic regression on both models' log-probs + simple stylometric features

In [ ]:
import glob, os, re, itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import f1_score, classification_report, ConfusionMatrixDisplay

SEED = 42
IN_COLAB = not os.path.exists("/kaggle/input")  # Kaggle also has a /content folder, so check for /kaggle/input instead
if IN_COLAB:
    # Colab: competition data in /content/data (same download cell as notebook 03), outputs of 02 and 03 in Drive
    from google.colab import drive, files
    drive.mount("/content/drive")
    SEARCH = "/content/drive/MyDrive/hw3_outputs"
    if not os.path.exists("/content/data/train.csv"):
        files.upload()  # kaggle.json
        os.makedirs("/root/.kaggle", exist_ok=True)
        os.replace("kaggle.json", "/root/.kaggle/kaggle.json"); os.chmod("/root/.kaggle/kaggle.json", 0o600)
        os.system("kaggle competitions download -c ai-generated-text-detection-hw3-ucu-ai-2026 -p /content/data")
        os.system("cd /content/data && unzip -oq *.zip")
    DATA_DIR = "/content/data"
    os.chdir(SEARCH)  # results are written next to the inputs in Drive
else:
    SEARCH = "/kaggle/input"
    DATA_DIR = os.path.dirname(glob.glob("/kaggle/input/**/train.csv", recursive=True)[0])
train = pd.read_csv(f"{DATA_DIR}/train.csv")
test = pd.read_csv(f"{DATA_DIR}/test.csv")
sample_sub = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")
TEXT_COL, LABEL_COL = "text", "label"
LANG_COL = next((c for c in ["language", "lang"] if c in train.columns), None)
DOMAIN_COL = "domain" if "domain" in train.columns else None

def make_split(train, label_col="label", lang_col=None, n_val=10_000, n_train=40_000, seed=42):
    strat = train[label_col].astype(str) + ("_" + train[lang_col].astype(str) if lang_col else "")
    rest, val = train_test_split(train, test_size=n_val, stratify=strat, random_state=seed)
    strat_rest = strat.loc[rest.index]
    train_sub, _ = train_test_split(rest, train_size=n_train, stratify=strat_rest, random_state=seed)
    return train_sub.reset_index(drop=True), val.reset_index(drop=True)

_, val = make_split(train, LABEL_COL, LANG_COL)
CLASSES = np.array(sorted(train[LABEL_COL].unique()))
y = val[LABEL_COL].map({c: i for i, c in enumerate(CLASSES)}).values
print("val", val.shape, "first val ids:", val["id"].head(3).tolist())

In [ ]:
def load(name, ids):
    """Find an output file of notebook 02/03 (added via Add Input) and align it to `ids`."""
    path = glob.glob(f"{SEARCH}/**/{name}", recursive=True)[0]
    df = pd.read_csv(path).set_index("id").loc[ids]
    return df[CLASSES].values

P_val = {"tfidf": load("baseline_val_proba.csv", val["id"])}
P_test = {"tfidf": load("baseline_test_proba.csv", test["id"])}
for mode in ["head", "headtail", "chunks"]:
    P_val[f"xlmr_{mode}"] = load(f"xlmr_val_proba_{mode}.csv", val["id"])
for mode in ["headtail", "chunks"]:
    P_test[f"xlmr_{mode}"] = load(f"xlmr_test_proba_{mode}.csv", test["id"])

for k, p in P_val.items():
    print(f"{k:15s} val macro-F1 = {f1_score(y, p.argmax(1), average='macro'):.4f}")

# reference XLM-R = best mode that also has test predictions
XL = max(["xlmr_headtail", "xlmr_chunks"], key=lambda k: f1_score(y, P_val[k].argmax(1), average="macro"))
print("reference model:", XL)

## Helpers: tuning functions + out-of-fold evaluation inside val

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
WEIGHTS = np.round(np.arange(0, 1.01, 0.1), 1)
OFFSETS = np.round(np.arange(-1.0, 1.51, 0.25), 2)

def blend(w, a, b):
    return w * a + (1 - w) * b

def fit_weight(pa, pb, yy):
    return max(WEIGHTS, key=lambda w: f1_score(yy, blend(w, pa, pb).argmax(1), average="macro"))

def apply_offsets(p, off):
    return np.log(p + 1e-9) + off

def fit_offsets(p, yy):
    # offset for class 0 fixed at 0; grid over the other two
    best, best_f = np.zeros(3), -1
    for o1, o2 in itertools.product(OFFSETS, OFFSETS):
        off = np.array([0, o1, o2])
        f = f1_score(yy, apply_offsets(p, off).argmax(1), average="macro")
        if f > best_f:
            best, best_f = off, f
    return best

def oof(fit_predict):
    """fit_predict(train_idx, test_idx) -> predicted class ids for test_idx. Returns out-of-fold predictions."""
    pred = np.zeros(len(y), dtype=int)
    for tr, te in skf.split(np.zeros(len(y)), y):
        pred[te] = fit_predict(tr, te)
    return pred

results = []
def log(name, pred, note=""):
    f = f1_score(y, pred, average="macro")
    per = f1_score(y, pred, average=None)
    results.append({"experiment": name, "macro_f1": f, **{f"f1_{c}": v for c, v in zip(CLASSES, per)}, "note": note})
    print(f"{name:35s} {f:.4f}  " + "  ".join(f"{c}={v:.3f}" for c, v in zip(CLASSES, per)))

log("REF: XLM-R alone", P_val[XL].argmax(1), XL)
log("TF-IDF LogReg alone", P_val["tfidf"].argmax(1))

## E1 — weighted blend of XLM-R and TF-IDF

In [ ]:
pa, pb = P_val[XL], P_val["tfidf"]
log("E1 blend (w tuned, OOF)", oof(lambda tr, te: blend(fit_weight(pa[tr], pb[tr], y[tr]), pa[te], pb[te]).argmax(1)))
curve = [f1_score(y, blend(w, pa, pb).argmax(1), average="macro") for w in WEIGHTS]
plt.figure(figsize=(5, 3)); plt.plot(WEIGHTS, curve, marker="o")
plt.xlabel("weight of XLM-R"); plt.ylabel("val macro-F1"); plt.title("Blend weight (whole val, for illustration)")
plt.tight_layout(); plt.savefig("blend_weight.png", dpi=120); plt.show()

## E2 — class offsets for macro-F1 (on XLM-R alone and on the blend)

In [ ]:
log("E2 offsets on XLM-R (OOF)", oof(lambda tr, te: apply_offsets(pa[te], fit_offsets(pa[tr], y[tr])).argmax(1)))

def blend_then_offsets(tr, te):
    w = fit_weight(pa[tr], pb[tr], y[tr])
    off = fit_offsets(blend(w, pa[tr], pb[tr]), y[tr])
    return apply_offsets(blend(w, pa[te], pb[te]), off).argmax(1)
log("E1+E2 blend + offsets (OOF)", oof(blend_then_offsets))

## E3 — stacking with stylometric features

In [ ]:
def stylo(text):
    text = text[:5000]
    words = text.split(); n_w = max(len(words), 1)
    sents = [s for s in re.split(r"[.!?]+", text) if s.strip()]
    sl = [len(s.split()) for s in sents] or [0]
    return [np.log1p(len(words)), np.mean([len(w) for w in words]) if words else 0, np.mean(sl), np.std(sl),
            len(set(w.lower() for w in words[:200])) / max(min(n_w, 200), 1),
            text.count(",") / n_w, text.count("\n") / n_w,
            sum(not c.isalnum() and not c.isspace() for c in text) / max(len(text), 1),
            sum(c.isdigit() for c in text) / max(len(text), 1), sum(c.isupper() for c in text) / max(len(text), 1)]
STYLO_NAMES = ["log_words", "avg_word_len", "avg_sent_len", "std_sent_len", "ttr200", "comma_pw", "newline_pw",
               "punct_pc", "digit_pc", "upper_pc"]

def onehot(df):
    # language + domain are given for test rows too, so they are legitimate input features.
    # EDA: the label mix depends strongly on domain (arxiv/gutenberg ~85-90% ai) and test has a different domain mix.
    cols = []
    for col in [LANG_COL, DOMAIN_COL]:
        if col:
            cols += [(df[col] == v).astype(float).values for v in sorted(train[col].unique())]
    return np.stack(cols, 1) if cols else np.zeros((len(df), 0))

S_val = np.array([stylo(t) for t in val[TEXT_COL]]); S_test = np.array([stylo(t) for t in test[TEXT_COL]])
lp = lambda p: np.log(p + 1e-6)
X_val_stack = {
    "probs only": np.hstack([lp(P_val[XL]), lp(P_val["tfidf"])]),
    "probs + stylo + lang/domain": np.hstack([lp(P_val[XL]), lp(P_val["tfidf"]), S_val, onehot(val)]),
    "stylo + lang/domain only": np.hstack([S_val, onehot(val)]),
}
make_meta = lambda: make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=2000))  # scaler fit inside each fold
for name, X in X_val_stack.items():
    log(f"E3 stacking: {name} (OOF)", oof(lambda tr, te: make_meta().fit(X[tr], y[tr]).predict(X[te])))

# same features, but a small gradient-boosting model that can learn interactions (e.g. length x domain)
make_gb = lambda: HistGradientBoostingClassifier(max_iter=200, learning_rate=0.05, max_leaf_nodes=15, random_state=SEED)
Xg = X_val_stack["probs + stylo + lang/domain"]
log("E3b stacking GB: probs + stylo + lang/domain (OOF)", oof(lambda tr, te: make_gb().fit(Xg[tr], y[tr]).predict(Xg[te])))

## Summary (all numbers: macro-F1 on the same 10k val split; tuned methods are out-of-fold)

In [ ]:
summary = pd.DataFrame(results)
summary["delta_vs_ref"] = summary["macro_f1"] - summary.loc[0, "macro_f1"]
summary.round(4).to_csv("ensemble_results.csv", index=False)
summary.round(4)

## Final model: best method, re-fitted on the full val split, applied to test

In [ ]:
candidates = {
    "REF: XLM-R alone": lambda: (P_test[XL].argmax(1), P_val[XL].argmax(1)),
    "E1 blend (w tuned, OOF)": lambda: (lambda w: (blend(w, P_test[XL], P_test["tfidf"]).argmax(1),
                                                   blend(w, pa, pb).argmax(1)))(fit_weight(pa, pb, y)),
    "E2 offsets on XLM-R (OOF)": lambda: (lambda off: (apply_offsets(P_test[XL], off).argmax(1),
                                                       apply_offsets(pa, off).argmax(1)))(fit_offsets(pa, y)),
}
def final_blend_offsets():
    w = fit_weight(pa, pb, y); off = fit_offsets(blend(w, pa, pb), y)
    print("weight", w, "offsets", off)
    return apply_offsets(blend(w, P_test[XL], P_test["tfidf"]), off).argmax(1), apply_offsets(blend(w, pa, pb), off).argmax(1)
candidates["E1+E2 blend + offsets (OOF)"] = final_blend_offsets
X_test_stack = {
    "probs only": np.hstack([lp(P_test[XL]), lp(P_test["tfidf"])]),
    "probs + stylo + lang/domain": np.hstack([lp(P_test[XL]), lp(P_test["tfidf"]), S_test, onehot(test)]),
    "stylo + lang/domain only": np.hstack([S_test, onehot(test)]),
}
for name in X_val_stack:
    def f(name=name):
        m = make_meta().fit(X_val_stack[name], y)
        return m.predict(X_test_stack[name]), m.predict(X_val_stack[name])
    candidates[f"E3 stacking: {name} (OOF)"] = f

def final_gb():
    name = "probs + stylo + lang/domain"
    m = make_gb().fit(X_val_stack[name], y)
    return m.predict(X_test_stack[name]), m.predict(X_val_stack[name])
candidates["E3b stacking GB: probs + stylo + lang/domain (OOF)"] = final_gb

best_name = summary.sort_values("macro_f1", ascending=False)["experiment"].iloc[0]
print("best experiment:", best_name)
test_pred, val_fit_pred = candidates[best_name]()

# confusion matrix of the out-of-fold version is the honest one; this is the in-sample one for reference
fig, ax = plt.subplots(figsize=(4.5, 4))
ConfusionMatrixDisplay.from_predictions(CLASSES[y], CLASSES[val_fit_pred], labels=CLASSES, normalize="true",
                                        values_format=".2f", ax=ax, colorbar=False)
ax.set_title("final model on val (row-normalised)"); plt.tight_layout(); plt.savefig("final_confusion.png", dpi=120); plt.show()

sub = sample_sub[["id"]].merge(pd.DataFrame({"id": test["id"], "label": CLASSES[test_pred]}), on="id", how="left")
assert sub["label"].notna().all() and len(sub) == len(sample_sub)
sub.to_csv("submission_ensemble.csv", index=False)
print(sub["label"].value_counts(normalize=True).round(3))